In [164]:
import torch
import torch.nn as nn

In [165]:

inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

### Simple self attention

In [166]:
cntx_vector=torch.zeros(inputs.shape[0],inputs.shape[1])
print(cntx_vector)        

tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])


In [167]:
for i,inp in enumerate(inputs):
    for j,inpi in enumerate(torch.softmax(inp @ torch.transpose(inputs,1,0),dim=0)):
        cntx_vector[i] +=inpi*inputs[j]

In [168]:
print(cntx_vector)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


### Self attention using learnable weights

In [169]:
d_out=8
d_model=inputs.shape[1]
torch.manual_seed(123)
wq=nn.Linear(d_model,d_out,bias=False)
wk=nn.Linear(d_model,d_out,bias=False)
wv=nn.Linear(d_model,d_out,bias=False)

In [170]:
# Query, Key and Value
Q=wq(inputs)
K=wk(inputs)
V=wv(inputs)
#Attention Score
att_score=Q@(K.T)
#attention weight
att_weight=torch.softmax(att_score/d_out**0.5,dim=-1)
print(att_weight.shape)
#Context vector
context_vactor=att_weight@V
print(context_vactor)

torch.Size([6, 6])
tensor([[ 0.5396,  0.2519, -0.5360, -0.0106,  0.0341, -0.2925, -0.3916,  0.6015],
        [ 0.5373,  0.2527, -0.5366, -0.0176,  0.0268, -0.2893, -0.3957,  0.6012],
        [ 0.5373,  0.2527, -0.5366, -0.0175,  0.0268, -0.2894, -0.3956,  0.6012],
        [ 0.5362,  0.2522, -0.5354, -0.0173,  0.0268, -0.2889, -0.3947,  0.5999],
        [ 0.5378,  0.2523, -0.5351, -0.0146,  0.0282, -0.2904, -0.3942,  0.6003],
        [ 0.5359,  0.2523, -0.5359, -0.0184,  0.0264, -0.2884, -0.3951,  0.6001]],
       grad_fn=<MmBackward0>)


### Casual attention mask

In [171]:
wq=nn.Linear(d_model,d_out,bias=False)
wk=nn.Linear(d_model,d_out,bias=False)
wv=nn.Linear(d_model,d_out,bias=False)
# Query, Key and Value
Q=wq(inputs)
K=wk(inputs)
V=wv(inputs)
#Attention Score
att_score=Q@(K.T)
casual_mask=torch.tril(torch.ones(att_score.shape))
casual_mask=casual_mask.masked_fill(casual_mask==0,-torch.inf)
print(casual_mask)
print(att_score)
masked_att_score=att_score + casual_mask
print(masked_att_score)
att_weight=torch.softmax(masked_att_score/d_out**0.5,dim=-1)
print(att_weight)
#attention weight
#Context vector
context_vactor=att_weight@V
print(context_vactor)

tensor([[1., -inf, -inf, -inf, -inf, -inf],
        [1., 1., -inf, -inf, -inf, -inf],
        [1., 1., 1., -inf, -inf, -inf],
        [1., 1., 1., 1., -inf, -inf],
        [1., 1., 1., 1., 1., -inf],
        [1., 1., 1., 1., 1., 1.]])
tensor([[-0.0798, -0.2023, -0.2042, -0.1088, -0.1819, -0.0973],
        [-0.7587, -0.8404, -0.8388, -0.4117, -0.5748, -0.4599],
        [-0.7455, -0.8255, -0.8243, -0.4033, -0.5711, -0.4474],
        [-0.5019, -0.5385, -0.5362, -0.2652, -0.3429, -0.3092],
        [-0.2970, -0.3243, -0.3302, -0.1396, -0.3433, -0.0963],
        [-0.6529, -0.7093, -0.7034, -0.3587, -0.3981, -0.4440]],
       grad_fn=<MmBackward0>)
tensor([[0.9202,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.2413, 0.1596,   -inf,   -inf,   -inf,   -inf],
        [0.2545, 0.1745, 0.1757,   -inf,   -inf,   -inf],
        [0.4981, 0.4615, 0.4638, 0.7348,   -inf,   -inf],
        [0.7030, 0.6757, 0.6698, 0.8604, 0.6567,   -inf],
        [0.3471, 0.2907, 0.2966, 0.6413, 0.6019, 0.5560]],
 

#### Multihead Attention using weight split

In [199]:
class causal_attention(nn.Module):
    def __init__(self,d_in,d_out,qkv_bias):
        super().__init__()
        torch.manual_seed(123)
        self.W_query=nn.Linear(d_in,d_out,qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,qkv_bias)
    def forward(self,x):
        query=self.W_query(x)
        key=self.W_key(x)
        value=self.W_value(x)
        #Attention Score
        attention_score=query @ (key.T)
        casual_mask=torch.tril(torch.ones(attention_score.shape))
        casual_att_score=casual_mask.masked_fill(casual_mask==0,-torch.inf)
        #attention weights
        attention_weight=torch.softmax(casual_att_score/(d_out**0.5),dim=-1)
        #context vector
        context_vector=attention_weight @ value
        return context_vector       

In [200]:
self_att=causal_attention(inputs.shape[1],8,False)
print(self_att(inputs))

tensor([[ 0.4566,  0.2729, -0.4303, -0.1992, -0.3749, -0.1810, -0.5684,  0.5063],
        [ 0.5909,  0.3038, -0.5869, -0.1034, -0.1349, -0.2898, -0.5345,  0.6644],
        [ 0.6355,  0.3125, -0.6337, -0.0653, -0.0539, -0.3272, -0.5200,  0.7132],
        [ 0.5730,  0.2794, -0.5866, -0.0554, -0.0112, -0.2982, -0.4530,  0.6524],
        [ 0.5594,  0.2541, -0.5148,  0.0239,  0.0180, -0.3095, -0.3976,  0.5959],
        [ 0.5363,  0.2514, -0.5339, -0.0142,  0.0293, -0.2897, -0.3925,  0.5988]],
       grad_fn=<MmBackward0>)
